## AgentBench - LTP 환경 재현: 모델 비교

**논문**: Liu et al., *AgentBench: Evaluating LLMs as Agents*, ICLR 2024 (Lateral Thinking Puzzles)

Colab에서 *런타임 유형 - GPU*로 실행하세요.


### 0. 환경 설정


In [ ]:
# Colab에서 실행 시: 저장소를 클론합니다.
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    REPO_URL = "https://github.com/gksmfly/Agentbench-Mini-Eval.git"
    if not os.path.exists("Agentbench-Mini-Eval"):
        !git clone {REPO_URL}
    %cd Agentbench-Mini-Eval/project_ltp
    !pip install -q requests pandas matplotlib numpy transformers accelerate bitsandbytes
else:
    # 로컬(맥/리눅스)에서 실행 시: 이미 project_ltp/notebooks 안에 있다고 가정
    os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None

In [ ]:
import json
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["axes.unicode_minus"] = False

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

DATA_FILE = ROOT / "data" / "sample_cases.jsonl"
GOLD_FILE = ROOT / "data" / "ltp_dev_gold.jsonl"
METRICS_FILE = ROOT / "results" / "metrics.csv"

samples = [json.loads(l) for l in open(DATA_FILE)]
print(f"toy 샘플 {len(samples)}개 로드 완료")

def get_key(name: str, required: bool = True) -> str:
    """우선순위: 이미 설정된 환경변수 -> Colab Secrets(발표 전에 미리 등록) -> getpass 수동 입력"""
    if os.environ.get(name):
        return os.environ[name]
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    if required:
        return getpass(f"{name}: ")
    return getpass(f"{name} (없으면 그냥 Enter): ")

# API 키값 -- 발표 직전에 여기 따옴표 안에 직접 채워넣고 실행하세요.
# !! 채운 뒤에는 이 파일을 절대 커밋/푸시하지 마세요 (git에 키가 그대로 남습니다) !!
HARDCODED_OPENAI_API_KEY = ""      # 예: "sk-..."

os.environ["OPENAI_API_KEY"] = HARDCODED_OPENAI_API_KEY or get_key("OPENAI_API_KEY")


### 1. 핵심 아이디어 최소 재현

본 실험은 THUDM/AgentBench 프레임워크(CLI+Docker)로 실행한 것 중 핵심 부분인 호스트 역할(스토리+진실을 알고 솔버의 질문에 Yes/No/Irrelevant로 답하기)을 물어봅니다.

- `gpt-3.5-turbo`, `gpt-4o` → OpenAI API 호출

- `claude-sonnet-5` → Anthropic Messages API 호출 (타 벤더 검증 후보)

- `llama-3.1-8b`, `vicuna-13b-local` → 로컬 GPU 구동

실행 시, 아래 `MODEL_TO_RUN` 값만 바꿔서 원하는 모델 실행


In [ ]:
from baseline import build_host_history, call_llm

def demo(index: int, model: str):
    entry = samples[index]
    question = entry["sample_question"]
    history = build_host_history(entry, question)
    if model == "llama-3.1-8b":
        import torch
        print("[GPU 확인]", torch.cuda.is_available(),
              torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(없음, CPU로 진행 시 매우 느릴 수 있음)")
    reply = call_llm(model, history)
    print(f"--- {model} ---\n{reply}\n")

MODEL_TO_RUN = "gpt-3.5-turbo"  # gpt-3.5-turbo / gpt-4o / claude-sonnet-5 / llama-3.1-8b / vicuna-13b-local

entry = samples[0]
print(f"[puzzle index] {entry['index']}")
print(f"[story] {entry['story']}\n")
print(f"[question put to the host] {entry['sample_question']}\n")

demo(0, MODEL_TO_RUN)

print(f"[gold host answer for reference, captured live by the framework] {entry.get('gold_host_answer')}")


## 2. 데이터 셋으로 본실험 결과 — 모델 비교

각 모델을 AgentBench 프레임워크(Docker + 실제 MySQL + 멀티라운드 에이전트 루프)로 전체 데이터 300문제에 대해 실행한 결과

`src/evaluate.py`로 독립 재검증 완료

- Game Progress & SGA by model: 얼마나 잘 맞혔는지
  
  모델별 GP(메인 지표)와 SGA(보조 지표)를 나란히 비교해서, 어떤 모델이 진실에 더 가깝게
  다가갔는지/개별 질문을 더 잘 맞혔는지 한눈에 보게 하는 겁니다.

- Run status breakdown: 왜 해당 점수가 나왔는지
  
  정답 여부를 떠나서, 각 게임이 완료(completed)로 끝났는지 라운드 초과(task limit reached)로
  끝났는지 비율


In [ ]:
metrics = pd.read_csv(METRICS_FILE)
metrics[metrics["metric"] == "GP"]

In [ ]:
models = ["llama-3.1-8b", "vicuna-13b-local", "claude-sonnet-5", "gpt-3.5-turbo", "gpt-4o"]

# (a) GP & SGA per model
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gp_sga = metrics[metrics["metric"].isin(["GP", "SGA"])]
pivot = gp_sga.pivot(index="metric", columns="model", values="value").reindex(models, axis=1)
pivot.plot.bar(ax=axes[0])
axes[0].set_title("Game Progress & SGA by model")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=0)

# (b) completion status breakdown per model
status = metrics[metrics["metric"].str.startswith("status:")].copy()
status["status"] = status["metric"].str.replace("status:", "")
status_pivot = status.pivot(index="model", columns="status", values="value").reindex(models).fillna(0)
status_pivot.plot.bar(stacked=True, ax=axes[1])
axes[1].set_title("Run status breakdown")
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=7, loc="upper right")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


### 2.1 추가 분석: 비용·속도·라운드사용량·응답 길이

GP/SGA 이외 안 보이는 실사용 관점 지표

- 비용: tiktoken으로 토큰 수 세서 공개된 가격표 곱해서 추정 (로컬 GPU 모델은 $0)

- 속도: 로그에 찍힌 타임스탬프 차이로 계산

- 라운드사용량: 게임당 실제 사용한 라운드 수(최대 25)

- 응답 길이: 모델 답변 글자 수 평균


In [ ]:
ANALYSIS_FILE = ROOT / "results" / "analysis.csv"
analysis = pd.read_csv(ANALYSIS_FILE)

key_metrics = [
    "estimated_cost_usd_dev_subset",
    "avg_rounds_used",
    "games_per_minute",
    "avg_agent_response_chars",
]
ltp_summary = analysis[analysis["metric"].isin(key_metrics)].pivot(
    index="metric", columns="model", values="value"
).reindex(models, axis=1)
display(ltp_summary)


### 2.2 추가 분석 시각화

- Estimated cost (dev subset): 게임 20개 도는 데 실제로 든 비용

- Average rounds used: 게임당 평균 몇 라운드를 쓰고 끝났는지 (최대 25) — 25에 가까울수록 대부분 라운드 초과로 종료됐다는 뜻


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ltp_summary.loc[["estimated_cost_usd_dev_subset"]].T.plot.bar(ax=axes[0], legend=False, color="darkorange")
axes[0].set_title("Estimated cost (dev subset, USD)")
axes[0].tick_params(axis="x", rotation=15)

ltp_summary.loc[["avg_rounds_used"]].T.plot.bar(ax=axes[1], legend=False)
axes[1].set_title("Average rounds used (max 25)")
axes[1].set_ylim(0, 25)
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()
